# Reprojection toolbox

Notebook workflow corresponding to `app.py`: open or create a Metashape project, reconstruct it, export overlapping images, run object detection, and reproject annotations.

In [ ]:
import os
import sys
from pathlib import Path

import torch  # Imported before Metashape, as in app.py.
import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

import Metashape
from reconstruction.reconstruct import reconstruct_model_metashape
from reprojection import annotations
from reprojection import camera_utils as cu
from reprojection import metashape_utils as mu
from reprojection import reprojection_database as rdb
from object_detection import fifty_one_utils as fou
from object_detection import inference
from object_detection import sahi_inference as sahi
from ultralytics import YOLO

print(f"Project root: {PROJECT_ROOT}")

## Define project paths

Set these values directly. `ANNOTATION_REPORT_PATH` may be left as `None` when the notebook should use the report generated by the inference cell.

In [ ]:
PROJECT_NAME = "my_project"
PROJECT_DIR = PROJECT_ROOT / "data" / PROJECT_NAME
IMAGE_DIR = PROJECT_DIR / "images"
METASHAPE_PROJECT_PATH = PROJECT_DIR / f"{PROJECT_NAME}.psx"
CHUNK_NAME = "Chunk 1"
MODEL_PATH = PROJECT_ROOT / "object_detection" / "yolo11n.pt"
ANNOTATION_REPORT_PATH = None

PROJECT_DIR = Path(PROJECT_DIR)
IMAGE_DIR = Path(IMAGE_DIR)
METASHAPE_PROJECT_PATH = Path(METASHAPE_PROJECT_PATH)
MODEL_PATH = Path(MODEL_PATH)
print(f"Project directory: {PROJECT_DIR}")

## Open or create the Metashape project

In [ ]:
metashape_path = METASHAPE_PROJECT_PATH
doc = Metashape.Document()

if metashape_path.exists():
    doc.open(str(metashape_path))
    chunk = next((candidate for candidate in doc.chunks if candidate.label == CHUNK_NAME), None)
    if chunk is None:
        raise ValueError(f"Chunk not found: {CHUNK_NAME}")
else:
    metashape_path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(str(metashape_path))
    doc.addChunk()
    chunk = doc.chunk
    chunk.label = CHUNK_NAME
    doc.save()

print(f"Chunk: {chunk.label}; cameras: {len(chunk.cameras)}")

## Reconstruction

Images are added only when the selected chunk has no cameras. The reconstruction sequence is the same as the GUI action.

In [ ]:
if len(chunk.cameras) == 0:
    image_dir = IMAGE_DIR
    if not image_dir.is_dir():
        raise NotADirectoryError(f"Image directory not found: {image_dir}")
    mu.add_images(chunk, str(image_dir))
    doc.save()

reconstruct_model_metashape(doc, chunk)
doc.save()
print("Reconstruction finished")

## Export overlapping images

In [ ]:
overlap_dir = PROJECT_DIR / "overlapping_images"
overlap_dir.mkdir(parents=True, exist_ok=True)
temporary_chunk = mu.get_overlapping_images(chunk, str(overlap_dir))
doc.remove([temporary_chunk])
doc.save()
print(f"Images exported to {overlap_dir}")

## Object detection inference

This follows `InferenceThread`: SAHI is selected when an image is more than twice the model input size; otherwise classic YOLO inference is used.

In [ ]:
model_path = MODEL_PATH
if not model_path.is_file():
    raise FileNotFoundError(f"Inference model not found: {model_path}")

dataset = fou.import_image_directory(str(overlap_dir), "overlapping_image_dataset")
if len(dataset) == 0:
    raise ValueError(f"No images found in {overlap_dir}")
dataset.compute_metadata()
model = YOLO(str(model_path))
image_size = model.model.args["imgsz"]
sample = dataset.first()
width, height = sample["metadata"]["width"], sample["metadata"]["height"]

if width > 2 * image_size or height > 2 * image_size:
    inferred_dataset = sahi.sahi_inference(str(model_path), dataset, slice=image_size)
else:
    inferred_dataset = inference.YOLO_inference(dataset, str(model_path))

report_path = PROJECT_DIR / "inference_report.csv"
pd.DataFrame(fou.fo_to_csv(inferred_dataset)).to_csv(report_path, index=False)
ANNOTATION_REPORT_PATH = report_path
print(f"Inference report written to {report_path}")

## Reprojection

Set `INCLUDE_IMAGE_LABELS` to add whole-image labels from the Metashape cameras. Set `RESET_DATABASE` to rebuild an existing reprojection database.

In [ ]:
INCLUDE_IMAGE_LABELS = False
RESET_DATABASE = False
db_dir = PROJECT_DIR / f"reprojection_{PROJECT_NAME}"
db_dir.mkdir(parents=True, exist_ok=True)
camera_reprojectors = cu.chunk_to_camera_reprojector(chunk, str(db_dir))

if RESET_DATABASE or not (db_dir / "reprojection.db").exists():
    tie_points = mu.get_all_tie_points(chunk, str(db_dir))
    report_path = ANNOTATION_REPORT_PATH
    if report_path:
        report = pd.read_csv(report_path)
        columns = ["filename", "label_name", "points"]
        if "confidence" in report.columns:
            columns.insert(2, "confidence")
        report = report[columns]
        if "confidence" not in report.columns:
            report["confidence"] = 1
        if INCLUDE_IMAGE_LABELS:
            report = pd.concat([report, cu.chunk_to_img_labels(chunk)], ignore_index=True)
    else:
        report = cu.chunk_to_img_labels(chunk)
    session = annotations.inference_report_to_reprojection_database(
        str(db_dir), report, camera_reprojectors, tie_points
    )
else:
    session, _ = rdb.open_reprojection_database_session(str(db_dir), False)

session = annotations.annotations_to_individuals(session, str(db_dir))
mu.plot_all_annotations(
    session, camera_reprojectors, chunk, PROJECT_NAME,
    pointify=True, confidence_threshold=0.5
)
doc.save()
print(f"Reprojection finished; database: {db_dir}")


## Save and release the document

Run this last, before opening the project in the desktop application.

In [ ]:
doc.save()
del chunk
del doc
print("Metashape document saved and released")